# AI-assisted country brief from Enterprise Surveys indicators

Type a country code and run every cell. The notebook:

1. asks the World Bank Data360 API for **every** Enterprise Surveys indicator (about 200
   national-level indicators across all survey topics) and collects them for all countries;
2. builds a table of facts for the country: latest value, change since the previous survey,
   and how it compares with the countries and groups you choose;
3. gives the table to an AI model, which picks the most interesting points and drafts them;
4. **checks every number the AI wrote against the table**, and drops any point that fails;
5. draws a one-page brief and writes an Excel workbook with the points, the data behind
   the figure, the full table and the checks.

**You choose in the first cell:** the country, the topics or indicators to look at (leave
them empty and the notebook searches everything), and what to compare with.

The AI model is the one built into Google Colab (`google.colab.ai`, no key needed).
Outside Colab, or if the model is not available, the points are written by rule instead.
Only public data are sent to the AI model.

In [ ]:
# ---- 1. Settings: the only cell you need to edit -------------------------------
COUNTRY = "GTM"          # three-letter ISO code, e.g. GTM, ETH, KEN, IND, VNM

# WHAT TO LOOK AT. Leave both lists empty to search every indicator for what stands out.
TOPICS = []              # e.g. ["Gender", "Finance"]: one point for each topic you list
INDICATORS = []          # e.g. ["GEND4", "FIN14"], or words from a name: ["woman top manager"]

# WHAT TO COMPARE WITH.
COMPARE_COUNTRIES = []   # e.g. ["HND", "SLV", "MEX"]
COMPARE_GROUPS = ["own region", "own income group"]
#   also allowed: "all countries", or any region or income group by name,
#   e.g. "Sub-Saharan Africa", "South Asia", "High income", "Lower middle income"
GROUP_STATISTIC = "average"   # "average" (how the Enterprise Surveys website compares groups) or "median"

USE_AI = True            # False = write the points by rule, without an AI model
FIRST_YEAR = 2010        # ignore surveys older than this
MAX_POINTS = 4           # how many points to make when TOPICS is empty

# Topic names you can use in TOPICS (the ones on enterprisesurveys.org):
#   Regulations, Taxes, Finance, Infrastructure and Climate, Gender, Trade,
#   Competition and Public Procurement, Corruption, Informality, Dispute Resolution, Jobs,
#   Workforce, Management Practices, Innovation and Technology, Firm Profile, Biggest Obstacle
# Indicator codes: run once, then open the "All indicators" sheet of the Excel file.

In [ ]:
# ---- 2. Find and collect every Enterprise Surveys indicator -------------------------
import datetime
import json
import re
import textwrap
import time
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import pandas as pd
import requests

plt.rcParams["font.family"] = "DejaVu Sans"   # ships with matplotlib, so the page looks the same everywhere
DATA360 = "https://data360api.worldbank.org/data360"     # indicators and data
WB_API = "https://api.worldbank.org/v2"                  # country list (region, income group)
COUNTRY = COUNTRY.strip().upper()

# Topic of each indicator, as listed in the Enterprise Surveys "Indicator Descriptions" document
# (26 October 2026, enterprisesurveys.org). The API has no topic field, so the topic is read from the
# code: WB_ES_T_FIN14 belongs to the family FIN. A few codes sit under a different topic from the rest of
# their family; those are listed one by one.
TOPIC_OF_FAMILY = {
    "REG": "Regulations", "BUS": "Regulations", "TAX": "Taxes", "FIN": "Finance",
    "IN": "Infrastructure and Climate", "GEND": "Gender", "TR": "Trade",
    "COMP": "Competition and Public Procurement", "CORR": "Corruption", "GRAFT": "Corruption",
    "INFOR": "Informality", "DISP": "Dispute Resolution", "CRIME": "Crime", "JOBS": "Jobs",
    "WK": "Workforce", "MGMT": "Management Practices", "T": "Innovation and Technology",
    "CAR": "Firm Profile", "LFORM": "Firm Profile", "PERF": "Firm Profile", "OBST": "Biggest Obstacle",
}
TOPIC_OF_CODE = {
    "REG2": "Taxes", "REG4": "Taxes", "REG5": "Taxes", "REG6": "Taxes",     # tax visits, tax constraints
    "T2": "Finance",                                                        # audited financial statement
    "T3": "Firm Profile",                                                   # capacity utilization
    "CRIME9": "Dispute Resolution", "CORR12": "Dispute Resolution",         # courts
}


def topic_of(code):
    """Topic of an indicator code such as WB_ES_T_BREADY_TAX4_MEDIAN (its core is TAX4)."""
    core = code.replace("WB_ES_T_", "").replace("BREADY_", "").split("_")[0]
    return TOPIC_OF_CODE.get(core, TOPIC_OF_FAMILY.get(re.match(r"[A-Z]*", core).group(), "Other"))


UNITS = {"Percentage": "%", "Percentage of firms": "%", "Days": " days", "Hours": " hours",
         "Weeks": " weeks", "Months": " months", "Years": " years"}


def call_api(url, params=None, body=None):
    """One call to a World Bank API (GET, or POST when a body is given), tried up to 4 times
    because the APIs are sometimes slow. Returns the decoded JSON."""
    waits = [5, 20, 45]                       # seconds to wait before the 2nd, 3rd and 4th try
    for attempt in range(len(waits) + 1):
        try:
            if body is None:
                r = requests.get(url, params=params, timeout=120)
            else:
                r = requests.post(url, json=body, timeout=120)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as err:
            if attempt == len(waits):
                raise RuntimeError("A World Bank API did not answer after 4 tries. It is probably busy: "
                                   "wait a few minutes, then click Runtime > Run all again.") from err
            print(f"    API slow ({type(err).__name__}); trying again in {waits[attempt]} seconds ...")
            time.sleep(waits[attempt])


def find_indicators():
    """Every national-level Enterprise Surveys indicator in Data360, with its name, unit and topic.
    Codes starting WB_ES_T_ cover all firms; WB_ES_M_ and WB_ES_S_ repeat them by sector."""
    answer = call_api(f"{DATA360}/searchv2", body={
        "count": True, "search": "*", "top": 1000, "skip": 0,
        "select": "series_description/idno, series_description/name, series_description/measurement_unit",
        "filter": "series_description/database_id eq 'WB_ES'"})
    rows = []
    for item in answer.get("value", []):
        d = item.get("series_description") or {}
        code = d.get("idno") or ""
        if not code.startswith("WB_ES_T_"):
            continue
        rows.append({"code": code, "short_code": code.replace("WB_ES_T_", ""),
                     "indicator": (d.get("name") or code).strip(),
                     "unit": UNITS.get(d.get("measurement_unit"), ""),
                     "topic": topic_of(code)})
    return pd.DataFrame(rows).sort_values("code").reset_index(drop=True)


def get_indicator(code):
    """National values for one indicator: all countries, all survey years.
    The three breakdown filters keep the all-firms figure and drop the size and sector splits."""
    rows, skip = [], 0
    while True:
        answer = call_api(f"{DATA360}/data", params={
            "DATABASE_ID": "WB_ES", "INDICATOR": code, "COMP_BREAKDOWN_1": "_T",
            "COMP_BREAKDOWN_2": "_T", "COMP_BREAKDOWN_3": "_T", "skip": skip})
        page = answer.get("value") or []
        rows += [{"code": code, "iso3": r["REF_AREA"], "year": int(float(r["TIME_PERIOD"])),
                  "value": float(r["OBS_VALUE"])}
                 for r in page if r.get("OBS_VALUE") not in (None, "")]
        skip += len(page)
        if not page or skip >= int(answer.get("count") or 0):
            return rows


def get_countries():
    """Country list with region and income group. Regional aggregates are dropped."""
    records = call_api(f"{WB_API}/country", params={"format": "json", "per_page": 400})[1]
    df = pd.DataFrame([
        {"iso3": r["id"], "country": r["name"], "region": r["region"]["value"].strip(),
         "income": r["incomeLevel"]["value"].strip()}
        for r in records
    ])
    return df[df["region"] != "Aggregates"].reset_index(drop=True)


print("Finding the Enterprise Surveys indicators ...")
indicators = find_indicators()
assert len(indicators) > 0, "The indicator list came back empty."
print(f"Found {len(indicators)} indicators in {indicators['topic'].nunique()} topics.")

print(f"Collecting {len(indicators)} indicators for all countries (about a minute) ...")
with ThreadPoolExecutor(max_workers=4) as pool:                 # four calls at a time
    pages = list(pool.map(get_indicator, indicators["code"]))
raw = pd.DataFrame([row for page in pages for row in page])
raw = raw[raw["year"] >= FIRST_YEAR]
countries = get_countries()
raw = raw.merge(countries, on="iso3", how="inner").merge(indicators, on="code")

assert COUNTRY in set(countries["iso3"]), f"'{COUNTRY}' is not a country code the API knows."
me = countries.set_index("iso3").loc[COUNTRY]
assert COUNTRY in set(raw["iso3"]), f"No Enterprise Surveys indicators found for {me['country']}."
assert not raw.duplicated(["code", "iso3", "year"]).any(), "More than one value for a country, indicator and year."
print(f"Collected {len(raw):,} values for {raw['iso3'].nunique()} countries.")
print(f"Country: {me['country']} | region: {me['region']} | income group: {me['income']}")

In [ ]:
# ---- 3. Read your choices: topics, indicators and comparators ------------------------
def pick_from(name, options, what):
    """Match what was typed to one of the options, ignoring capitals; part of a name is enough."""
    typed = name.strip().lower()
    exact = [o for o in options if o.lower() == typed]
    partial = [o for o in options if typed in o.lower()]
    if len(exact) == 1 or len(partial) == 1:
        return (exact or partial)[0]
    raise ValueError(f"'{name}' is not a {what} I can match"
                     + (f" (it fits several: {', '.join(partial)})" if partial else "")
                     + f". Choose from: {', '.join(sorted(options))}.")


# Topics
all_topics = sorted(indicators["topic"].unique())
topics = []
for t in TOPICS:
    match = pick_from(t, all_topics, "topic")
    if match not in topics:
        topics.append(match)

# Indicators: a code such as GEND4, or words from the indicator's name
chosen_codes = []
for entry in INDICATORS:
    typed = entry.strip()
    by_short_code = indicators[indicators["short_code"].str.upper().isin([typed.upper(), typed.upper().replace("WB_ES_T_", "")])]
    by_name = indicators[indicators["indicator"].str.contains(typed, case=False, regex=False)]
    found = by_short_code if len(by_short_code) else by_name
    if found.empty:
        raise ValueError(f"No indicator matches '{entry}'. Use a code such as GEND4, or words from the name. "
                         "The full list is in the 'All indicators' sheet of the Excel file from an earlier run.")
    if len(found) > 8:
        print(f"  '{entry}' matches {len(found)} indicators; keeping the first 8. Use a code to be exact.")
        found = found.head(8)
    for code, name in zip(found["code"], found["indicator"]):
        if code not in chosen_codes:
            chosen_codes.append(code)
            print(f"  '{entry}' -> {code.replace('WB_ES_T_', '')}: {name}")

# Comparators: other countries, and groups of countries (their average or median)
STAT = GROUP_STATISTIC.strip().lower()
assert STAT in ("average", "median"), 'GROUP_STATISTIC must be "average" or "median".'
regions, incomes = sorted(countries["region"].unique()), sorted(countries["income"].unique())
COMPARATORS = []          # each: label, kind ("country" or "group"), and how to find its countries
for code in COMPARE_COUNTRIES:
    code = code.strip().upper()
    if code == COUNTRY:
        continue
    if code not in set(countries["iso3"]):
        raise ValueError(f"'{code}' in COMPARE_COUNTRIES is not a country code the API knows.")
    COMPARATORS.append({"label": countries.set_index("iso3").loc[code, "country"], "kind": "country", "iso3": code})
for g in COMPARE_GROUPS:
    typed = g.strip().lower()
    if typed in ("own region", "region"):
        column, name = "region", me["region"]
    elif typed in ("own income group", "own income", "income", "income group"):
        column, name = "income", me["income"]
    elif typed in ("all countries", "all", "world", "all economies"):
        column, name = None, "All countries"
    else:
        name = pick_from(g, regions + incomes, "region or income group")
        column = "region" if name in regions else "income"
    label = f"{name} {STAT}"
    if label not in [c["label"] for c in COMPARATORS]:
        COMPARATORS.append({"label": label, "kind": "group", "column": column, "name": name})
assert COMPARATORS, "Choose at least one comparator in COMPARE_COUNTRIES or COMPARE_GROUPS."

print("Looking at:   " + (", ".join(topics) if topics else "")
      + (" + " if topics and chosen_codes else "")
      + (f"{len(chosen_codes)} chosen indicator(s)" if chosen_codes else "")
      + ("every indicator" if not topics and not chosen_codes else ""))
print("Compared with: " + ", ".join(c["label"] for c in COMPARATORS))

In [ ]:
# ---- 4. Build the table of facts for the country ------------------------------------
facts = []
for code, d in raw.sort_values("year").groupby("code"):
    own = d[d["iso3"] == COUNTRY]
    if own.empty:
        continue
    latest = d.groupby("iso3", as_index=False).tail(1)          # each country's most recent survey
    value, year, unit = own["value"].iloc[-1], int(own["year"].iloc[-1]), own["unit"].iloc[0]
    f = {"code": code, "topic": own["topic"].iloc[0], "indicator": own["indicator"].iloc[0], "unit": unit.strip(),
         "value": value, "year": year, "previous_value": None, "previous_year": None, "change": None}
    if len(own) > 1:
        f["previous_value"], f["previous_year"] = own["value"].iloc[-2], int(own["year"].iloc[-2])
        f["change"] = value - f["previous_value"]

    # One column per comparator, and the gap to it (value minus comparator)
    for c in COMPARATORS:
        if c["kind"] == "country":
            other = latest[latest["iso3"] == c["iso3"]]
            f[c["label"]] = other["value"].iloc[0] if len(other) else None
            f[c["label"] + " year"] = int(other["year"].iloc[0]) if len(other) else None
        else:
            group = latest if c["column"] is None else latest[latest[c["column"]] == c["name"]]
            f[c["label"]] = (group["value"].mean() if STAT == "average" else group["value"].median()) \
                if len(group) else None
        f["gap vs " + c["label"]] = value - f[c["label"]] if f[c["label"]] is not None else None

    # Rule-based flags: how unusual is this indicator for the country, within its own region?
    reg = latest[latest["region"] == me["region"]]
    f["region_rank"], f["region_countries"] = int((reg["value"] > value).sum()) + 1, len(reg)
    f["obstacle_rank"] = None
    flags, n = [], len(reg)
    position = (f["region_rank"] - 1) / (n - 1) if n >= 5 else 0.5        # 0 = highest, 1 = lowest
    if n >= 5 and position <= 0.15:
        flags.append("among the highest in the region")
    if n >= 5 and position >= 0.85:
        flags.append("among the lowest in the region")
    relative_change = 0.0
    if f["change"] is not None:
        relative_change = abs(f["change"]) / max(abs(f["previous_value"]), 1.0)
        if relative_change >= 0.30 and (unit != "%" or abs(f["change"]) >= 5):
            flags.append("large change since the previous survey")
        ratio = f["value"] / f["previous_value"] if f["previous_value"] > 0 else None
        if ratio is not None and (ratio >= 2 or ratio <= 0.5) and abs(f["change"]) >= 5:
            flags.append("more than doubled or halved since the previous survey: verify before using")
    f["flags"] = flags
    f["signal"] = abs(position - 0.5) * 2 + min(relative_change, 1.0)       # used to order the table
    facts.append(f)

facts = pd.DataFrame(facts).sort_values("signal", ascending=False).reset_index(drop=True)
latest_survey = int(facts["year"].max())               # the country's most recent survey year
for i in facts.index[facts["year"] < latest_survey]:
    facts.at[i, "flags"] = facts.at[i, "flags"] + [f"not in the {latest_survey} survey: value is from an older survey"]

# Biggest obstacle: each firm names one, so rank the obstacles within the country (1 = named most often)
is_obstacle = (facts["topic"] == "Biggest Obstacle") & (facts["year"] == latest_survey)
facts.loc[is_obstacle, "obstacle_rank"] = facts.loc[is_obstacle, "value"].rank(ascending=False, method="min")
if is_obstacle.any():
    obstacle_total = facts.loc[is_obstacle, "value"].sum()
    print(f"Biggest-obstacle shares add up to {obstacle_total:.1f}% "
          + ("(as expected)." if 97 <= obstacle_total <= 103 else "- WARNING: expected about 100%."))

comparator_columns = [c["label"] for c in COMPARATORS]
gap_columns = ["gap vs " + c for c in comparator_columns]
number_columns = ["value", "previous_value", "change"] + comparator_columns + gap_columns
facts[number_columns] = facts[number_columns].astype(float).round(1)

# The indicators this run looks at: your topics and indicators, or everything
if topics or chosen_codes:
    selected = facts[facts["topic"].isin(topics) | facts["code"].isin(chosen_codes)].reset_index(drop=True)
    missing = [c.replace("WB_ES_T_", "") for c in chosen_codes if c not in set(facts["code"])]
    if missing:
        print(f"No value for {me['country']} on: {', '.join(missing)}.")
    assert len(selected), f"{me['country']} has no data for the topics and indicators you chose."
else:
    selected = facts
print(f"{len(facts)} indicators have a value for {me['country']}; this run looks at {len(selected)} of them.")
print("\nThe most unusual, by rule:")
print(selected[["topic", "indicator", "year", "value", "previous_value"] + comparator_columns[:2]]
      .head(12).to_string(index=False, max_colwidth=55))

In [ ]:
# ---- 5. Ask the AI model for the most interesting points ------------------------------
def clean(v):
    """Make a value safe for JSON: no NaN, no numpy types."""
    if isinstance(v, list):
        return v
    if v is None or pd.isna(v):
        return None
    return v.item() if hasattr(v, "item") else v


whole_number_columns = ["previous_year", "obstacle_rank"] + [c["label"] + " year" for c in COMPARATORS if c["kind"] == "country"]
records = [{k: clean(v) for k, v in row.items() if k != "signal"} for row in selected.to_dict("records")]
for r in records:                                   # years and ranks are whole numbers, not 2017.0
    for key in whole_number_columns:
        r[key] = int(r[key]) if r[key] is not None else None

# The table goes to the model, one line per indicator
COLUMNS = [c for c in selected.columns if c != "signal"]


def as_line(r):
    cells = ["; ".join(r[c]) if c == "flags" else "" if r[c] is None else str(r[c]).strip() for c in COLUMNS]
    return " | ".join(cells)


TABLE = "\n".join([" | ".join(COLUMNS)] + [as_line(r) for r in records])
if topics:
    n_points = len(topics)
    TASK = (f"Write exactly one point for each of these topics: {', '.join(topics)}. For each topic, choose its "
            "most interesting indicator, or a pair of indicators that tell a story together.")
elif chosen_codes:
    n_points = min(MAX_POINTS, len(records))
    TASK = (f"These indicators were chosen by the reader. Write {n_points} points about them, covering as many "
            "of the indicators as you can, and say how the country compares on each.")
else:
    n_points = MAX_POINTS
    TASK = (f"Read every line. Choose the {n_points} most interesting points for a reader who knows the region. "
            "Prefer values far from the comparators, large changes since the previous survey, and above all pairs "
            "of indicators that tell a story together or seem to contradict each other. Cover at least three "
            "different topics.")

PROMPT = f"""You are helping an economist write a short country brief from World Bank Enterprise Surveys indicators.

Country: {me['country']}. Region: {me['region']}. Income group: {me['income']}.

DATA: one line per indicator, {len(records)} indicators, columns separated by " | ". Every number you may use is here.
{TABLE}

Column guide: "value" is the latest survey value and "year" its survey year. "previous_value" and
"previous_year" are the survey before it, and "change" is value minus previous_value. A column ending in
"{STAT}" is the simple {STAT} of the latest values of the countries in that group. A column named after a country
is that country's latest value, and the column ending in "year" beside it is its survey year. A column
starting "gap vs" is value minus that comparator. "region_rank" is the country's position among
"region_countries" countries in its own region, where 1 is the highest value. "obstacle_rank" ranks the
biggest-obstacle indicators within the country, where 1 is the obstacle firms name most often. "flags" are
notes produced by simple rules. Empty cells mean no data.

TASK: {TASK}

RULES:
- Use only numbers that appear in DATA, written exactly as they appear or rounded to a whole number.
  Do not calculate any new number.
- Describe what the data show. Do not claim causes.
- If a change looks too large to be plausible between two survey rounds, say in "verify" what should be
  checked. Otherwise set "verify" to an empty string.
- Plain English. Each "text" is at most 45 words. Do not put indicator codes in the text.

Return only JSON: a list of objects with the keys "headline" (at most 8 words), "text",
"indicators" (the list of indicator codes the point uses) and "verify"."""
print(f"The prompt holds {len(records)} indicators and is {len(PROMPT):,} characters long.")


def ask_ai(prompt):
    """Send the prompt to the AI model built into Google Colab. Returns (answer, None) or (None, reason)."""
    try:
        from google.colab import ai
    except ImportError:
        return None, "not running in Google Colab"
    try:
        return ai.generate_text(prompt), None
    except Exception as err:                      # quota, sign-in or network problems
        return None, f"the AI model did not answer ({type(err).__name__})"


def parse_points(answer):
    """Pull the JSON list out of the AI's answer. Returns [] if it cannot be read."""
    try:
        points = json.loads(answer[answer.index("["):answer.rindex("]") + 1])
    except (ValueError, TypeError, AttributeError):
        return []
    good = []
    for p in points if isinstance(points, list) else []:
        if isinstance(p, dict) and p.get("headline") and p.get("text"):
            codes = p.get("indicators") if isinstance(p.get("indicators"), list) else []
            good.append({"headline": str(p["headline"]).strip(), "text": str(p["text"]).strip(),
                         "indicators": [str(c) for c in codes], "verify": str(p.get("verify") or "").strip()})
    return good


ai_answer, ai_problem = (None, "USE_AI is set to False")
if USE_AI:
    print("Asking the AI model ...")
    ai_answer, ai_problem = ask_ai(PROMPT)
ai_points = parse_points(ai_answer) if ai_answer else []
if ai_answer and not ai_points:
    ai_problem = "the AI's answer could not be read as JSON"
print(f"The AI model proposed {len(ai_points)} point(s)." if ai_points else f"No AI points: {ai_problem}.")

In [ ]:
# ---- 6. Check every number the AI wrote against the table ------------------------------
def numbers_in(text):
    """Every number written in a piece of text, as floats. '1,200' is read as 1200."""
    return [float(m.replace(",", "")) for m in re.findall(r"(?<![\w.])\d[\d,]*(?:\.\d+)?", text)]


def allowed_numbers(rows):
    """Numbers the AI may use for these indicators: the table's values, or the same rounded."""
    allowed = set()
    for row in rows:
        for v in row.values():
            if isinstance(v, (int, float)) and not isinstance(v, bool):
                allowed.update({abs(v), float(round(abs(v)))})
    return allowed


def matches(x, allowed):
    return any(abs(x - a) < 0.051 for a in allowed)


by_code = {r["code"]: r for r in records}
check_log, kept = [], []
for p in ai_points:
    unknown = [c for c in p["indicators"] if c not in by_code]
    # A number passes only if it is in the table rows of the indicators the point says it uses
    own_numbers = allowed_numbers([by_code[c] for c in p["indicators"] if c in by_code])
    written = numbers_in(" ".join([p["headline"], p["text"], p["verify"]]))
    not_found = [x for x in written if not matches(x, own_numbers)]
    passed = not unknown and not not_found and bool(p["indicators"])
    check_log.append({"headline": p["headline"], "numbers_written": written, "numbers_not_in_table": not_found,
                      "unknown_indicator_codes": unknown, "passed": passed})
    if passed:
        kept.append(p)
    print(f"{'PASS' if passed else 'DROP'}  {p['headline']}: {len(written)} number(s) written, "
          f"{len(not_found)} not in the rows of the indicators it lists"
          + (f", codes not in this run's table {unknown}" if unknown else ""))


def rule_point(r):
    """One point written by rule from one line of the table."""
    first = COMPARATORS[0]["label"]
    name, u = r["indicator"].split(" (")[0], r["unit"] if r["unit"] == "%" else " " + r["unit"] if r["unit"] else ""
    text = f"{name}: {r['value']}{u} in {r['year']}"
    if r[first] is not None:
        text += f", against {r[first]}{u} for {first.replace(' ' + STAT, ' (' + STAT + ')')}"
    text += "."
    if r["previous_value"] is not None:
        text += f" It was {r['previous_value']}{u} in {r['previous_year']}."
    verify = "Large change between survey rounds: check the question and sample." \
        if any("verify" in f for f in r["flags"]) else ""
    return {"headline": name if len(name) <= 70 else name[:67] + "...", "text": text,
            "indicators": [r["code"]], "verify": verify, "source": "rule"}


def rule_points(k):
    """Points written by rule from the top of the table (the most unusual indicators first):
    one per topic, or one per indicator when you chose the indicators yourself."""
    out, topics_used = [], set()
    for r in records:
        if len(out) == k or (r["topic"] in topics_used and not (chosen_codes and not topics)):
            continue
        topics_used.add(r["topic"])
        out.append(rule_point(r))
    return out


def main_topic(p):
    codes = [c for c in p["indicators"] if c in by_code]
    return by_code[codes[0]]["topic"] if codes else None


for p in kept:
    p["source"] = "ai"
if topics:
    # One point for each topic you listed: the AI's first sound point on it, or a rule-written one
    points = []
    for t in topics:
        from_ai = [p for p in kept if main_topic(p) == t]
        from_table = [r for r in records if r["topic"] == t]
        if from_ai:
            points.append(from_ai[0])
        elif from_table:
            points.append(rule_point(from_table[0]))
        else:
            print(f"No data for {me['country']} in the topic {t}.")
elif kept:
    points = kept[:n_points]
else:
    points = rule_points(n_points)
n_ai = sum(p["source"] == "ai" for p in points)
written_by = "ai" if n_ai == len(points) else "rule" if n_ai == 0 else "ai and rule"
total = sum(len(numbers_in(" ".join([p["headline"], p["text"], p["verify"]]))) for p in points if p["source"] == "ai")
print(f"\n{len(kept)} of {len(ai_points)} AI point(s) passed the checks. The brief uses {n_ai} AI point(s) "
      f"and {len(points) - n_ai} written by rule.")

json.dump({"country": COUNTRY, "date": str(datetime.date.today()), "written_by": written_by,
           "settings": {"TOPICS": TOPICS, "INDICATORS": INDICATORS, "COMPARE_COUNTRIES": COMPARE_COUNTRIES,
                        "COMPARE_GROUPS": COMPARE_GROUPS, "GROUP_STATISTIC": GROUP_STATISTIC},
           "ai_problem": ai_problem, "prompt": PROMPT, "ai_answer": ai_answer, "checks": check_log,
           "points_used": points},
          open(f"ai_log_{COUNTRY}.json", "w"), indent=1)

for p in points:
    print(f"\n{p['headline']}\n  {p['text']}" + (f"\n  Check before using: {p['verify']}" if p["verify"] else ""))

In [ ]:
# ---- 7. Draw the brief --------------------------------------------------------------------
SURFACE, INK, INK2, MUTED, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#c3c2b7"
ACCENT, ACCENT_LIGHT = "#2a78d6", "#a9c9ee"            # the country now, and in its previous survey
OTHER_COUNTRY, GROUP = "#8f8e86", "#c3c2b7"            # comparator countries, and groups of countries
W = 8.27                                               # page width in inches (A4)


def wrap(text, width):
    return textwrap.wrap(text, width) or [""]


def fmt(v, u):
    return f"{v:.1f}%" if u == "%" else f"{v:.1f} {u}".strip()


def topics_of(p):
    """Topics of the indicators a point uses, in order, without repeats."""
    seen = []
    for c in p["indicators"]:
        if c in by_code and by_code[c]["topic"] not in seen:
            seen.append(by_code[c]["topic"])
    return " · ".join(seen).upper()


def bars_for(r):
    """The bars of one panel: the country, its previous survey, then each comparator that has a value."""
    bars = [(f"{me['country']} {r['year']}", r["value"], ACCENT)]
    if r["previous_value"] is not None:
        bars.append((f"{me['country']} {r['previous_year']}", r["previous_value"], ACCENT_LIGHT))
    for c in COMPARATORS:
        if r[c["label"]] is None:
            continue
        if c["kind"] == "country":
            bars.append((f"{c['label']} {r[c['label'] + ' year']}", r[c["label"]], OTHER_COUNTRY))
        else:
            bars.append((c["label"], r[c["label"]], GROUP))
    return bars


# Indicators to chart: the ones you chose by name, then those the points use (each point's first one first)
shown = [c for c in chosen_codes if c in by_code]
for position in range(6):
    for p in points:
        codes = [c for c in p["indicators"] if c in by_code]
        if position < len(codes) and codes[position] not in shown:
            shown.append(codes[position])
shown = shown[:8]
panel_rows = (len(shown) + 1) // 2
most_bars = max(len(bars_for(by_code[c])) for c in shown)
bar_area = 0.28 * most_bars                            # height of the bars in a panel, in inches
panel_height = 0.6 + bar_area + 0.3                    # title, bars, gap below

looked_at = (f"the topics {', '.join(topics)}" if topics else "") + (" and " if topics and chosen_codes else "") \
    + (f"{len(chosen_codes)} chosen indicator(s)" if chosen_codes else "")
subtitle = wrap(f"World Bank Enterprise Surveys indicators for {me['country']}, compared with "
                f"{', '.join(c['label'].replace(' ' + STAT, '') for c in COMPARATORS)}. "
                + (f"This brief looks at {looked_at}: {len(records)} indicators."
                   if looked_at else
                   f"The points below were chosen from {len(records)} indicators across "
                   f"{selected['topic'].nunique()} topics."), 100)
if written_by == "rule":
    credit = f"Points written by rule from the indicator table ({ai_problem or 'no AI point passed the checks'})."
else:
    credit = (f"Points drafted by an AI model in Google Colab from the indicator table, then checked by code: "
              f"all {total} numbers in the AI's text match the data.")
    if written_by == "ai and rule":
        n_rule = len(points) - n_ai
        credit += (" One point had no sound AI text and was written by rule." if n_rule == 1
                   else f" {n_rule} points had no sound AI text and were written by rule.")
note = wrap(f"Source: World Bank Enterprise Surveys, through the World Bank Data360 API "
            f"(data360api.worldbank.org), retrieved {datetime.date.today():%d %B %Y}. Group figures are simple "
            f"{STAT}s of each country's most recent survey since {FIRST_YEAR}, calculated here; they are not "
            f"official World Bank aggregates. Survey years differ between countries. {credit}", 125)

# Work out the page height from the amount of text and the number of panels, so nothing overlaps
blocks = [(p, wrap(p["headline"], 80), wrap(p["text"], 100),
           wrap("Check before using: " + p["verify"], 100) if p["verify"] else []) for p in points]
text_height = sum(0.19 + 0.25 * len(h) + 0.185 * (len(t) + len(v)) + 0.19 for _, h, t, v in blocks)
H = 0.55 + 0.42 + 0.19 * len(subtitle) + 0.25 + text_height + 0.25 + panel_height * panel_rows \
    + 0.16 * len(note) + 0.5
fig = plt.figure(figsize=(W, H), facecolor=SURFACE)
y = H - 0.55                                           # distance from the bottom of the page, in inches


def put(x, y_in, text, **kw):
    fig.text(x / W, y_in / H, text, ha="left", va="top", **kw)


put(0.58, y, f"{me['country']}: what stands out", fontsize=19, fontweight="bold", color=INK)
y -= 0.42
put(0.58, y, "\n".join(subtitle), fontsize=9, color=INK2, linespacing=1.4)
y -= 0.19 * len(subtitle) + 0.25
for p, headline_lines, text_lines, verify_lines in blocks:
    put(0.58, y, topics_of(p), fontsize=7, color=MUTED)
    y -= 0.19
    put(0.58, y, "\n".join(headline_lines), fontsize=10.5, fontweight="bold", color=INK, linespacing=1.3)
    y -= 0.25 * len(headline_lines)
    put(0.58, y, "\n".join(text_lines), fontsize=9, color=INK2, linespacing=1.4)
    y -= 0.185 * len(text_lines)
    if verify_lines:
        put(0.58, y, "\n".join(verify_lines), fontsize=9, color=MUTED, style="italic", linespacing=1.4)
        y -= 0.185 * len(verify_lines)
    y -= 0.19
y -= 0.25

figure_rows = []                                       # the data behind the figure, for the Excel file
for i, code in enumerate(shown):
    r = by_code[code]
    bars = bars_for(r)
    top = y - panel_height * (i // 2)                  # top of this panel, in inches
    left = 2.48 if i % 2 == 0 else 6.24
    height = 0.28 * len(bars)
    ax = fig.add_axes([left / W, (top - 0.6 - height) / H, 1.45 / W, height / H], facecolor=SURFACE)
    values = [v for _, v, _ in bars]
    low, high = min(0, min(values)), max(0, max(values))
    span = max(high - low, 1e-9)
    ax.set_xlim(low - (0.3 * span if low < 0 else 0), high + 0.3 * span)
    for k, (label, v, colour) in enumerate(bars):
        yy = len(bars) - 1 - k
        ax.barh(yy, v, height=0.56, color=colour)
        ax.text(v + 0.025 * span * (1 if v >= 0 else -1), yy, fmt(v, r["unit"]), va="center",
                ha="left" if v >= 0 else "right", fontsize=8, color=INK,
                fontweight="bold" if k == 0 else "normal")
        ax.text(-0.04, yy, "\n".join(wrap(label, 22)), transform=ax.get_yaxis_transform(), va="center",
                ha="right", fontsize=7.5, color=INK if k == 0 else INK2, linespacing=0.95)
        figure_rows.append({"Topic": r["topic"], "Indicator": r["indicator"], "Unit": r["unit"],
                            "Bar": label, "Value": v})
    ax.set_ylim(-0.6, len(bars) - 0.4)
    ax.set_xticks([]); ax.set_yticks([])
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)
    ax.axvline(0, color=AXIS, linewidth=1)             # the zero line; bars to its left are negative
    title = wrap(r["indicator"], 46)
    if len(title) > 3:                                 # very long names are cut to three lines
        title = title[:2] + [title[2][:43].rstrip() + "..."]
    put(left - 1.9, top, "\n".join(title), fontsize=8.5, color=INK, linespacing=1.3)

put(0.58, 0.16 * len(note) + 0.3, "\n".join(note), fontsize=7, color=MUTED, linespacing=1.4)
fig.savefig(f"brief_{COUNTRY}.png", dpi=170, facecolor=SURFACE)
fig.savefig(f"brief_{COUNTRY}.pdf", facecolor=SURFACE)
plt.show()

In [ ]:
# ---- 8. Write the Excel workbook ------------------------------------------------------------
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

HEADERS = {"code": "Code", "topic": "Topic", "indicator": "Indicator", "unit": "Unit",
           "value": f"{me['country']} (latest)", "year": "Survey year", "previous_value": "Previous survey",
           "previous_year": "Previous survey year", "change": "Change", "region_rank": "Rank in own region (1 = highest)",
           "region_countries": "Countries in own region", "obstacle_rank": "Rank among obstacles (1 = named most)",
           "flags": "Flags"}
WIDE = {"Indicator": 60, "Text": 70, "Check before using": 45, "Flags": 45, "Headline": 34, "Indicators used": 50,
        "Numbers written": 30, "Bar": 34}


def add_sheet(wb, title, df, intro=None):
    """Write a table to a sheet: optional intro lines, bold header, filter, frozen header, sensible widths."""
    ws = wb.create_sheet(title)
    for i, line in enumerate(intro or [], start=1):
        ws.cell(row=i, column=1, value=line)
    if intro:
        ws["A1"].font = Font(bold=True, size=14)
    header_row = len(intro) + 2 if intro else 1        # one empty row between the intro and the table
    for j, name in enumerate(df.columns, start=1):
        ws.cell(row=header_row, column=j, value=name)
    for i, row in enumerate(df.itertuples(index=False), start=header_row + 1):
        for j, v in enumerate(row, start=1):
            if isinstance(v, list):
                v = "; ".join(map(str, v))
            elif v is None or (not isinstance(v, str) and pd.isna(v)):
                v = None
            elif hasattr(v, "item"):
                v = v.item()
            ws.cell(row=i, column=j, value=v)
    for j, name in enumerate(df.columns, start=1):
        cell = ws.cell(row=header_row, column=j)
        cell.font, cell.fill = Font(bold=True), PatternFill("solid", fgColor="E1E0D9")
        cell.alignment = Alignment(wrap_text=True, vertical="top")
        longest = max([len(str(name))] + [len(str(v)) for v in df[name].head(200)]) if len(df) else len(str(name))
        ws.column_dimensions[get_column_letter(j)].width = WIDE.get(name, min(max(longest + 2, 10), 28))
        for i in range(header_row + 1, ws.max_row + 1):
            c = ws.cell(row=i, column=j)
            if name in WIDE:
                c.alignment = Alignment(wrap_text=True, vertical="top")
            elif isinstance(c.value, float):
                c.number_format = "0.0"
    ws.freeze_panes = ws.cell(row=header_row + 1, column=1)
    ws.auto_filter.ref = f"A{header_row}:{get_column_letter(len(df.columns))}{ws.max_row}"
    return ws


def tidy(df):
    """Facts table with readable headings, short codes and no working columns."""
    out = df.drop(columns=["signal"], errors="ignore").copy()
    out["code"] = out["code"].str.replace("WB_ES_T_", "", regex=False)
    return out.rename(columns=HEADERS)


wb = Workbook()
wb.remove(wb.active)

# Sheet 1: the points
names = {r["code"]: r["indicator"] for r in records}
brief = pd.DataFrame([{"Topic": topics_of(p).title().replace(" And ", " and "), "Headline": p["headline"], "Text": p["text"],
                       "Check before using": p["verify"],
                       "Indicators used": "; ".join(names.get(c, c) for c in p["indicators"])} for p in points])
add_sheet(wb, "Brief", brief, intro=[
    f"{me['country']}: what stands out",
    f"World Bank Enterprise Surveys indicators. Produced on {datetime.date.today():%d %B %Y}.",
    "Looked at: " + (looked_at if looked_at else f"every indicator ({len(records)})"),
    "Compared with: " + ", ".join(c["label"] for c in COMPARATORS),
    credit])

# Sheet 2: the data behind the figure
add_sheet(wb, "Figure data", pd.DataFrame(figure_rows))

# Sheet 3: the full table for the indicators this run looked at
add_sheet(wb, "Indicators", tidy(selected.sort_values(["topic", "signal"], ascending=[True, False])))

# Sheet 4: the three most unusual indicators in every topic, by rule (all topics, whatever you chose)
by_topic = facts.sort_values("signal", ascending=False).groupby("topic").head(3).sort_values(
    ["topic", "signal"], ascending=[True, False])
add_sheet(wb, "By topic", tidy(by_topic))

# Sheet 5: every indicator, so you can pick codes for INDICATORS
catalogue = indicators.assign(has_data=indicators["code"].isin(facts["code"]).map({True: "yes", False: "no"}))
catalogue = catalogue[["short_code", "topic", "indicator", "unit", "has_data"]].sort_values(["topic", "short_code"])
catalogue.columns = ["Code", "Topic", "Indicator", "Unit", f"Has data for {me['country']}"]
add_sheet(wb, "All indicators", catalogue)

# Sheet 6: the checks on the AI's text
checks = pd.DataFrame([{"Headline": c["headline"], "Result": "kept" if c["passed"] else "dropped",
                        "Numbers written": ", ".join(f"{x:g}" for x in c["numbers_written"]),
                        "Not in the table": ", ".join(f"{x:g}" for x in c["numbers_not_in_table"]),
                        "Codes not in the table": ", ".join(c["unknown_indicator_codes"])} for c in check_log],
                      columns=["Headline", "Result", "Numbers written", "Not in the table", "Codes not in the table"])
add_sheet(wb, "AI checks", checks, intro=[
    "Checks on the AI model's text", credit, f"The full prompt and answer are in ai_log_{COUNTRY}.json."])

# Sheet 7: the figure itself
try:
    from openpyxl.drawing.image import Image
    picture = Image(f"brief_{COUNTRY}.png")
    picture.width, picture.height = picture.width * 0.45, picture.height * 0.45
    wb.create_sheet("Figure").add_image(picture, "A1")
except Exception as err:                               # the picture is optional
    print(f"The figure was not added to the workbook ({type(err).__name__}).")

wb.save(f"brief_{COUNTRY}.xlsx")
print(f"Saved brief_{COUNTRY}.xlsx (sheets: {', '.join(wb.sheetnames)}), brief_{COUNTRY}.png, "
      f"brief_{COUNTRY}.pdf and ai_log_{COUNTRY}.json")

In [ ]:
# ---- 9. Download the files (Colab only) ---------------------------------------------------
try:
    from google.colab import files
    for name in (f"brief_{COUNTRY}.xlsx", f"brief_{COUNTRY}.pdf", f"brief_{COUNTRY}.png", f"ai_log_{COUNTRY}.json"):
        files.download(name)
except ImportError:
    print("Not running in Colab: the files are in the current folder.")